# Ingest — Food Inspections (Chicago API)

Incremental download from the Socrata API (`:*,*` includes system fields `:id`, `:created_at`, `:updated_at`, `:version`), driven by a **global watermark** on `:updated_at`.

* **`max_rows`** — configurable widget (default 50000): caps how many records this run fetches. Override it in the notebook widget, or pass it as a parameter when running from Jobs / ETL pipelines / cron (`max_rows=<n>`).
* Output: `raw/<ingest-date>/food_inspections.csv` in the `foodguard_ai.raw.food_inspections` volume.
* Watermark: `watermark.txt` at the volume root (`foodguard_ai.raw.food_inspections`) — one global value, always the latest `:updated_at` across all dates; first run (no watermark) fetches from the beginning.

In [0]:
import csv
import io
from datetime import date
import requests
from pathlib import Path

# --- Configuration: overridable via Jobs task parameters (max_rows=<n>) ---
dbutils.widgets.text("max_rows", "50000")
MAX_ROWS = int(dbutils.widgets.get("max_rows") or "50000")

API_URL = "https://data.cityofchicago.org/resource/4ijn-s7e5.csv"

RAW_BASE = "/Volumes/foodguard_ai/raw/food_inspections"  # the foodguard_ai.raw.food_inspections volume

# One CSV per ingest date: raw/<ingest-date>/food_inspections.csv
INGEST_DATE = date.today().isoformat()
OUTPUT_FILE = Path(f"{RAW_BASE}/{INGEST_DATE}/food_inspections.csv")

# One GLOBAL watermark at the volume root — always the latest :updated_at seen
# across ALL date folders (never per-date).
WATERMARK_FILE = Path(f"{RAW_BASE}/watermark.txt")

PAGE_SIZE = 50_000  # Socrata max page size
DEFAULT_WATERMARK = "1970-01-01T00:00:00"  # first run: fetch from the beginning


def load_watermark() -> str:
    if WATERMARK_FILE.exists():
        return WATERMARK_FILE.read_text().strip()
    return DEFAULT_WATERMARK


def save_watermark(value: str) -> None:
    WATERMARK_FILE.write_text(value)


def fetch_page(watermark: str, offset: int, limit: int) -> str:
    """Fetch one page of rows created/updated at or after the watermark."""
    params = {
        "$select": ":*,*",                                # all columns + system fields (:id, :created_at, :updated_at, :version)
        "$where": f":updated_at >= '{watermark}'",        # e.g. :updated_at >= '2026-11-23T00:00:00'
        "$order": ":updated_at, :id",
        "$limit": limit,
        "$offset": offset,
    }
    response = requests.get(API_URL, params=params, timeout=120)
    response.raise_for_status()
    return response.text


def download_incremental() -> None:
    watermark = load_watermark()
    print(f"Watermark: {watermark} | max_rows: {MAX_ROWS}")

    all_rows = []
    max_updated_at = watermark
    offset = 0

    while len(all_rows) < MAX_ROWS:
        page_limit = min(PAGE_SIZE, MAX_ROWS - len(all_rows))
        page_text = fetch_page(watermark, offset, page_limit)
        rows = list(csv.DictReader(io.StringIO(page_text)))
        print(f"Page at offset {offset}: {len(rows)} rows")
        all_rows.extend(rows)
        page_max = max((r.get(":updated_at") or "" for r in rows), default="")
        max_updated_at = max(max_updated_at, page_max)

        if len(rows) < page_limit:   # short page = last page of available data
            break
        offset += page_limit

    if not all_rows:
        print("No new or updated rows — nothing to write.")
        return

    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = list(all_rows[0].keys())
    with OUTPUT_FILE.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)

    if max_updated_at > watermark:
        save_watermark(max_updated_at)

    print("Download completed!")
    print(f"File: {OUTPUT_FILE} ({len(all_rows)} rows, {len(fieldnames)} columns incl. :id/:created_at/:updated_at/:version)")
    print(f"New watermark: {max_updated_at}")


download_incremental()